# Tutorial 09 — Object Detection using SSD and YOLO

## Objective

This notebook implements **Tutorial 09: Object Detection using SSD and YOLO** using PyTorch.

The tutorial covers:

- Understanding SSD backbone and prediction heads
- Understanding YOLO backbone and detection head
- Building simple SSD-style and YOLO-style models from scratch
- Testing model output shapes using dummy data
- Using the custom object detection dataset developed in Tutorial 08
- Training/testing a pretrained YOLO model on the custom dataset
- Training/testing a pretrained SSD model on the custom dataset

Custom classes:

- Ororon
- furina

Expected folder structure:

```text
Tutorial_09/
├── tutorial_09_ssd_yolo_object_detection.ipynb
├── dataset/
│   ├── train/
│   │   ├── images/
│   │   └── labels/
│   ├── valid/
│   │   ├── images/
│   │   └── labels/
│   ├── test/
│   │   ├── images/
│   │   └── labels/
│   └── data.yaml
└── images/
```


## 0. NumPy Compatibility Check

If NumPy 2.x causes issues with Ultralytics/PyTorch on your setup, this cell installs NumPy 1.26.4.

If it installs NumPy, restart the kernel and run the notebook again from the start.


In [1]:
import sys
import subprocess

try:
    import numpy as np
    print("Current NumPy version:", np.__version__)
    np_major = int(np.__version__.split(".")[0])

    if np_major >= 2:
        print("NumPy 2.x detected. Installing NumPy 1.26.4...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "numpy==1.26.4",
            "--force-reinstall"
        ])

        raise SystemExit("NumPy was changed. Restart the kernel and run again.")
    else:
        print("NumPy version is compatible.")
except Exception as e:
    print("NumPy check error:", e)


Current NumPy version: 1.26.4
NumPy version is compatible.


## 1. Import Required Libraries

In [2]:
import os
import sys
import time
import random
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageDraw

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision.transforms import functional as F

# Install ultralytics if missing
try:
    import ultralytics
    from ultralytics import YOLO
    ULTRALYTICS_AVAILABLE = True
except ImportError:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
        import ultralytics
        from ultralytics import YOLO
        ULTRALYTICS_AVAILABLE = True
    except Exception as e:
        print("Ultralytics could not be installed.")
        print("Reason:", e)
        ULTRALYTICS_AVAILABLE = False

# Try disabling Ultralytics pin memory to avoid Windows DataLoader issues.
if ULTRALYTICS_AVAILABLE:
    try:
        import ultralytics.data.build as ultralytics_data_build
        ultralytics_data_build.PIN_MEMORY = False
        print("Ultralytics PIN_MEMORY disabled.")
    except Exception as e:
        print("Could not modify Ultralytics PIN_MEMORY:", e)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

if ULTRALYTICS_AVAILABLE:
    print("Ultralytics version:", ultralytics.__version__)

os.makedirs("images", exist_ok=True)


Ultralytics PIN_MEMORY disabled.
PyTorch version: 2.5.1+cu121
Torchvision version: 0.20.1+cu121
CUDA available: True
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
Ultralytics version: 8.4.51


## 2. Dataset Path

The same YOLO dataset from Tutorial 08 is used.

The dataset should contain:

```text
dataset/train/images
dataset/train/labels
dataset/valid/images
dataset/valid/labels
dataset/test/images
dataset/test/labels
dataset/data.yaml
```


In [3]:
DATASET_DIR = Path("dataset")
DATA_YAML = DATASET_DIR / "data.yaml"

if not DATASET_DIR.exists():
    raise FileNotFoundError("Dataset folder not found. Place the Tutorial 08 dataset folder here as 'dataset'.")

if not DATA_YAML.exists():
    raise FileNotFoundError("data.yaml not found. Expected: dataset/data.yaml")

print("Dataset folder found:", DATASET_DIR)
print("data.yaml found:", DATA_YAML)


Dataset folder found: dataset
data.yaml found: dataset\data.yaml


## 3. Read Dataset Information

In [4]:
import yaml

with open(DATA_YAML, "r", encoding="utf-8") as f:
    original_yaml = yaml.safe_load(f)

print("Original data.yaml:")
print(original_yaml)

class_names = original_yaml["names"]
num_classes = original_yaml["nc"]

fixed_yaml = {
    "path": str(DATASET_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": num_classes,
    "names": class_names
}

FIXED_DATA_YAML = DATASET_DIR / "data_fixed.yaml"

with open(FIXED_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(fixed_yaml, f, sort_keys=False)

print("\nFixed YOLO data yaml:")
print(fixed_yaml)

print("\nClasses:", class_names)
print("Number of classes:", num_classes)


Original data.yaml:
{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 2, 'names': ['Ororon', 'furina'], 'roboflow': {'workspace': 'bakas-workspace-5yl3w', 'project': 'tutorial-8', 'version': 1, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/bakas-workspace-5yl3w/tutorial-8/dataset/1'}}

Fixed YOLO data yaml:
{'path': 'C:\\Users\\mumer\\Downloads\\T9\\dataset', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 2, 'names': ['Ororon', 'furina']}

Classes: ['Ororon', 'furina']
Number of classes: 2


## 4. Count Dataset Images and Labels

In [5]:
def count_files(folder, extensions):
    folder = Path(folder)
    count = 0
    for ext in extensions:
        count += len(list(folder.glob(f"*{ext}")))
    return count

summary_rows = []

for split in ["train", "valid", "test"]:
    image_dir = DATASET_DIR / split / "images"
    label_dir = DATASET_DIR / split / "labels"

    summary_rows.append({
        "Split": split,
        "Images": count_files(image_dir, [".jpg", ".jpeg", ".png"]),
        "Labels": count_files(label_dir, [".txt"])
    })

dataset_summary_df = pd.DataFrame(summary_rows)
dataset_summary_df


,Split,Images,Labels
0,train,102,102
1,valid,29,29
2,test,14,14


# Part A — SSD Model from Scratch

SSD uses:

- a CNN backbone for feature extraction
- classification heads for class prediction
- box regression heads for bounding-box coordinates
- multi-scale feature maps for detecting objects at different sizes

This part builds a simplified SSD-style model and checks output shapes using dummy data.


## 5. Simple SSD Backbone and Prediction Head

In [6]:
class SimpleSSDBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        feature_maps = []

        x = self.block1(x)
        feature_maps.append(x)

        x = self.block2(x)
        feature_maps.append(x)

        x = self.block3(x)
        feature_maps.append(x)

        return feature_maps


class SimpleSSDHead(nn.Module):
    def __init__(self, in_channels, num_classes, num_boxes=4):
        super().__init__()

        self.num_classes = num_classes
        self.num_boxes = num_boxes

        self.class_head = nn.Conv2d(
            in_channels,
            num_boxes * num_classes,
            kernel_size=3,
            padding=1
        )

        self.box_head = nn.Conv2d(
            in_channels,
            num_boxes * 4,
            kernel_size=3,
            padding=1
        )

    def forward(self, feature_map):
        batch_size = feature_map.shape[0]

        class_pred = self.class_head(feature_map)
        box_pred = self.box_head(feature_map)

        class_pred = class_pred.permute(0, 2, 3, 1).contiguous()
        box_pred = box_pred.permute(0, 2, 3, 1).contiguous()

        class_pred = class_pred.view(batch_size, -1, self.num_classes)
        box_pred = box_pred.view(batch_size, -1, 4)

        return class_pred, box_pred


class SimpleSSD(nn.Module):
    def __init__(self, num_classes=3, num_boxes=4):
        super().__init__()

        self.backbone = SimpleSSDBackbone()

        self.heads = nn.ModuleList([
            SimpleSSDHead(32, num_classes, num_boxes),
            SimpleSSDHead(64, num_classes, num_boxes),
            SimpleSSDHead(128, num_classes, num_boxes)
        ])

    def forward(self, x):
        feature_maps = self.backbone(x)

        class_outputs = []
        box_outputs = []

        for feature_map, head in zip(feature_maps, self.heads):
            class_pred, box_pred = head(feature_map)
            class_outputs.append(class_pred)
            box_outputs.append(box_pred)

        class_outputs = torch.cat(class_outputs, dim=1)
        box_outputs = torch.cat(box_outputs, dim=1)

        return class_outputs, box_outputs


# SSD has background class, so add +1
ssd_scratch_num_classes = num_classes + 1

simple_ssd_model = SimpleSSD(num_classes=ssd_scratch_num_classes, num_boxes=4).to(device)

print(simple_ssd_model)


SimpleSSD(
  (backbone): SimpleSSDBackbone(
    (block1): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (block2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (block3): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
  )
  (heads): ModuleList(
    (0): SimpleSSDHead(
      (class_head): Conv2d(32, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (box_head): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    )
    (1): SimpleSSDHead(
      (class_head): Conv2d(64, 12, kernel_size=(3, 3), stride=(1, 

## 6. Test Simple SSD Output Shapes with Dummy Data

In [7]:
dummy_ssd_input = torch.randn(1, 3, 300, 300).to(device)

with torch.no_grad():
    ssd_class_pred, ssd_box_pred = simple_ssd_model(dummy_ssd_input)

print("SSD class prediction shape:", ssd_class_pred.shape)
print("SSD box prediction shape:", ssd_box_pred.shape)

print("\nInterpretation:")
print("Class output: batch_size x total_anchor_boxes x num_classes")
print("Box output: batch_size x total_anchor_boxes x 4")


SSD class prediction shape: torch.Size([1, 117976, 3])
SSD box prediction shape: torch.Size([1, 117976, 4])

Interpretation:
Class output: batch_size x total_anchor_boxes x num_classes
Box output: batch_size x total_anchor_boxes x 4


# Part B — YOLO Model from Scratch

YOLO divides the image into a grid. Each grid cell predicts:

- bounding box coordinates
- objectness/confidence
- class probabilities

This simplified YOLO-style model outputs:

```text
batch_size x grid_size x grid_size x (num_boxes * (5 + num_classes))
```


## 7. Simple YOLO Backbone and Detection Head

In [8]:
class SimpleYOLOBackbone(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(256, 512, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.features(x)


class SimpleYOLO(nn.Module):
    def __init__(self, num_classes=2, num_boxes=3):
        super().__init__()

        self.num_classes = num_classes
        self.num_boxes = num_boxes
        self.output_channels = num_boxes * (5 + num_classes)

        self.backbone = SimpleYOLOBackbone()
        self.head = nn.Conv2d(512, self.output_channels, kernel_size=1)

    def forward(self, x):
        x = self.backbone(x)
        x = self.head(x)

        # Convert from B x C x H x W to B x H x W x C
        x = x.permute(0, 2, 3, 1).contiguous()

        return x


simple_yolo_model = SimpleYOLO(num_classes=num_classes, num_boxes=3).to(device)

print(simple_yolo_model)


SimpleYOLO(
  (backbone): SimpleYOLOBackbone(
    (features): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): ReLU()
      (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (7): ReLU()
      (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (9): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (10): ReLU()
      (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (12): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): ReLU()
    )
  )
  (head): Conv2d(512, 21, kernel_size=(1, 1), stride=(1, 1))
)


## 8. Test Simple YOLO Output Shape with Dummy Data

In [9]:
dummy_yolo_input = torch.randn(1, 3, 416, 416).to(device)

with torch.no_grad():
    yolo_output = simple_yolo_model(dummy_yolo_input)

print("YOLO output shape:", yolo_output.shape)

print("\nInterpretation:")
print("Output: batch_size x grid_h x grid_w x [num_boxes * (5 + num_classes)]")
print("5 = x, y, w, h, confidence")
print("num_classes =", num_classes)


YOLO output shape: torch.Size([1, 26, 26, 21])

Interpretation:
Output: batch_size x grid_h x grid_w x [num_boxes * (5 + num_classes)]
5 = x, y, w, h, confidence
num_classes = 2


# Part C — Pretrained YOLOv8 Training and Testing

A pretrained YOLO model is fine-tuned on the custom dataset.

Base model:

```text
yolov8n.pt
```

This model starts from pretrained weights and then learns the custom classes:

- Ororon
- furina


## 9. Train Pretrained YOLOv8n on Custom Dataset

In [10]:
if not ULTRALYTICS_AVAILABLE:
    raise ImportError("Ultralytics is not available. Install using: pip install ultralytics")

YOLO_EPOCHS = 20
YOLO_IMAGE_SIZE = 640
YOLO_BATCH_SIZE = 8

yolo_model = YOLO("yolov8n.pt")

yolo_results = yolo_model.train(
    data=str(FIXED_DATA_YAML),
    epochs=YOLO_EPOCHS,
    imgsz=YOLO_IMAGE_SIZE,
    batch=YOLO_BATCH_SIZE,
    seed=SEED,
    workers=0,
    project="runs/tutorial_09",
    name="yolov8n_custom",
    exist_ok=True
)


Ultralytics 8.4.51  Python-3.10.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset\data_fixed.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_custom, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_

## 10. Validate and Test YOLOv8n Model

In [11]:
# Find YOLOv8n trained weights automatically
possible_yolo_weights = list(Path(".").rglob("yolov8n_custom/weights/best.pt"))

if len(possible_yolo_weights) == 0:
    possible_yolo_weights = list(Path(".").rglob("yolov8n_custom/weights/last.pt"))

print("Found YOLO weights:")
for p in possible_yolo_weights:
    print(p)

if len(possible_yolo_weights) == 0:
    raise FileNotFoundError("No YOLOv8n weights found. Training may not have completed.")

yolo_trained_path = possible_yolo_weights[0]
print("Using YOLO model:", yolo_trained_path)

yolo_trained_model = YOLO(str(yolo_trained_path))

yolo_test_metrics = yolo_trained_model.val(
    data=str(FIXED_DATA_YAML),
    split="test",
    imgsz=YOLO_IMAGE_SIZE,
    workers=0,
    project="runs/tutorial_09",
    name="yolov8n_test_results",
    exist_ok=True
)

print(yolo_test_metrics)


Found YOLO weights:
runs\detect\runs\tutorial_09\yolov8n_custom\weights\best.pt
Using YOLO model: runs\detect\runs\tutorial_09\yolov8n_custom\weights\best.pt
Ultralytics 8.4.51  Python-3.10.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 73 layers, 3,006,038 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 8.18.0 MB/s, size: 86.9 KB)
val: Scanning C:\Users\mumer\Downloads\T9\dataset\test\labels... 14 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 14/14 348.2it/s 0.0s
val: New cache created: C:\Users\mumer\Downloads\T9\dataset\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.3it/s 0.4s
                   all         14         14      0.921          1      0.988      0.678
                Ororon          8          8      0.858          1      0.982       0.63
                furina          6          6      0.985   

## 11. Visualize YOLOv8n Test Predictions

In [12]:
test_images_dir = DATASET_DIR / "test" / "images"

test_image_paths = sorted(
    list(test_images_dir.glob("*.jpg")) +
    list(test_images_dir.glob("*.jpeg")) +
    list(test_images_dir.glob("*.png"))
)

selected_test_images = test_image_paths[:8]

yolo_predictions = yolo_trained_model.predict(
    source=[str(p) for p in selected_test_images],
    imgsz=YOLO_IMAGE_SIZE,
    conf=0.25,
    save=False
)

plt.figure(figsize=(16, 8))

for i, result in enumerate(yolo_predictions):
    plotted = result.plot()[:, :, ::-1]

    plt.subplot(2, 4, i + 1)
    plt.imshow(plotted)
    plt.title(f"Image {i + 1}")
    plt.axis("off")

plt.suptitle("YOLOv8n Custom Model Test Predictions")
plt.tight_layout()
plt.savefig("images/tutorial_09_yolov8n_test_predictions.png", dpi=300, bbox_inches="tight")
plt.show()



0: 640x640 1 Ororon, 1 furina, 6.1ms
1: 640x640 1 furina, 6.1ms
2: 640x640 1 furina, 6.1ms
3: 640x640 1 furina, 6.1ms
4: 640x640 1 furina, 6.1ms
5: 640x640 1 furina, 6.1ms
6: 640x640 1 Ororon, 6.1ms
7: 640x640 1 Ororon, 6.1ms
Speed: 1.8ms preprocess, 6.1ms inference, 1.6ms postprocess per image at shape (1, 3, 640, 640)


<Figure size 1600x800 with 8 Axes>

# Part D — Pretrained SSD Training and Testing

A pretrained SSD300 VGG16 model from Torchvision is used.

The model is adapted for the custom dataset by replacing the classification head so that it predicts:

- background
- Ororon
- furina

The background class is required for SSD-style detection models.


## 12. Create PyTorch Detection Dataset from YOLO Labels

In [13]:
class YOLODetectionDataset(Dataset):
    def __init__(self, dataset_dir, split, image_size=300):
        self.dataset_dir = Path(dataset_dir)
        self.split = split
        self.image_size = image_size

        self.image_dir = self.dataset_dir / split / "images"
        self.label_dir = self.dataset_dir / split / "labels"

        self.image_paths = sorted(
            list(self.image_dir.glob("*.jpg")) +
            list(self.image_dir.glob("*.jpeg")) +
            list(self.image_dir.glob("*.png"))
        )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label_path = self.label_dir / (image_path.stem + ".txt")

        image = Image.open(image_path).convert("RGB")
        image = image.resize((self.image_size, self.image_size))
        image_tensor = F.to_tensor(image)

        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            for line in lines:
                parts = line.strip().split()

                if len(parts) != 5:
                    continue

                class_id = int(float(parts[0]))
                x_center = float(parts[1]) * self.image_size
                y_center = float(parts[2]) * self.image_size
                width = float(parts[3]) * self.image_size
                height = float(parts[4]) * self.image_size

                x1 = x_center - width / 2
                y1 = y_center - height / 2
                x2 = x_center + width / 2
                y2 = y_center + height / 2

                x1 = max(0, min(self.image_size - 1, x1))
                y1 = max(0, min(self.image_size - 1, y1))
                x2 = max(0, min(self.image_size - 1, x2))
                y2 = max(0, min(self.image_size - 1, y2))

                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])

                    # Torchvision detection models use 0 as background.
                    # Therefore, custom class IDs are shifted by +1.
                    labels.append(class_id + 1)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }

        return image_tensor, target


def detection_collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)


ssd_train_dataset = YOLODetectionDataset(DATASET_DIR, "train", image_size=300)
ssd_valid_dataset = YOLODetectionDataset(DATASET_DIR, "valid", image_size=300)
ssd_test_dataset = YOLODetectionDataset(DATASET_DIR, "test", image_size=300)

ssd_train_loader = DataLoader(
    ssd_train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,
    collate_fn=detection_collate_fn
)

ssd_test_loader = DataLoader(
    ssd_test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    collate_fn=detection_collate_fn
)

print("SSD training samples:", len(ssd_train_dataset))
print("SSD validation samples:", len(ssd_valid_dataset))
print("SSD test samples:", len(ssd_test_dataset))


SSD training samples: 102
SSD validation samples: 29
SSD test samples: 14


## 13. Load Pretrained SSD300 and Replace Classification Head

In [14]:
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead

def get_ssd_classification_head_channels(model):
    in_channels = []

    for layer in model.head.classification_head.module_list:
        conv_layer = None

        # Depending on torchvision version, the item can be Conv2d or Sequential.
        for module in layer.modules():
            if isinstance(module, nn.Conv2d):
                conv_layer = module
                break

        if conv_layer is None:
            raise RuntimeError("Could not find Conv2d layer inside SSD classification head.")

        in_channels.append(conv_layer.in_channels)

    return in_channels


SSD_NUM_CLASSES = num_classes + 1  # background + custom classes

ssd_pretrained_model = ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT)

# Replace classification head for custom number of classes.
anchor_generator = ssd_pretrained_model.anchor_generator
num_anchors = anchor_generator.num_anchors_per_location()
in_channels = get_ssd_classification_head_channels(ssd_pretrained_model)

ssd_pretrained_model.head.classification_head = SSDClassificationHead(
    in_channels=in_channels,
    num_anchors=num_anchors,
    num_classes=SSD_NUM_CLASSES
)

ssd_pretrained_model = ssd_pretrained_model.to(device)

print("SSD300 pretrained model loaded and adapted.")
print("SSD number of classes including background:", SSD_NUM_CLASSES)
print("Custom class mapping:")
print("0 = background")
for i, name in enumerate(class_names, start=1):
    print(f"{i} = {name}")


Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to C:\Users\mumer/.cache\torch\hub\checkpoints\ssd300_vgg16_coco-b556d3b4.pth
100.0%


SSD300 pretrained model loaded and adapted.
SSD number of classes including background: 3
Custom class mapping:
0 = background
1 = Ororon
2 = furina


## 14. Train Pretrained SSD on Custom Dataset

In [15]:
SSD_EPOCHS = 5
SSD_LR = 0.001

params = [p for p in ssd_pretrained_model.parameters() if p.requires_grad]
optimizer = torch.optim.Adam(params, lr=SSD_LR)

ssd_loss_history = []

start_time = time.time()

for epoch in range(SSD_EPOCHS):
    ssd_pretrained_model.train()

    epoch_loss = 0.0
    batches = 0

    for images, targets in ssd_train_loader:
        images = [img.to(device) for img in images]
        targets = [
            {k: v.to(device) for k, v in target.items()}
            for target in targets
        ]

        loss_dict = ssd_pretrained_model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()
        batches += 1

    avg_loss = epoch_loss / batches
    ssd_loss_history.append(avg_loss)

    print(f"Epoch {epoch + 1}/{SSD_EPOCHS} | SSD Training Loss: {avg_loss:.4f}")

ssd_training_time = time.time() - start_time
print("SSD training time:", round(ssd_training_time, 2), "seconds")


Epoch 1/5 | SSD Training Loss: 15.0948
Epoch 2/5 | SSD Training Loss: 6.5182
Epoch 3/5 | SSD Training Loss: 5.7144
Epoch 4/5 | SSD Training Loss: 5.5559
Epoch 5/5 | SSD Training Loss: 5.0857
SSD training time: 19.64 seconds


## 15. Plot SSD Training Loss

In [16]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(ssd_loss_history) + 1), ssd_loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("SSD300 Training Loss on Custom Dataset")
plt.grid(True)
plt.savefig("images/tutorial_09_ssd_training_loss.png", dpi=300, bbox_inches="tight")
plt.show()


<Figure size 800x500 with 1 Axes>

## 16. Test SSD Model on Custom Dataset

In [17]:
ssd_pretrained_model.eval()

ssd_prediction_images = []
ssd_prediction_outputs = []

with torch.no_grad():
    for images, targets in ssd_test_loader:
        images_device = [img.to(device) for img in images]
        outputs = ssd_pretrained_model(images_device)

        for img, output in zip(images, outputs):
            ssd_prediction_images.append(img.cpu())
            ssd_prediction_outputs.append({
                "boxes": output["boxes"].cpu(),
                "labels": output["labels"].cpu(),
                "scores": output["scores"].cpu()
            })

        if len(ssd_prediction_images) >= 8:
            break

print("SSD predictions generated:", len(ssd_prediction_images))


SSD predictions generated: 8


## 17. Visualize SSD Test Predictions

In [18]:
def draw_ssd_prediction(image_tensor, output, class_names, threshold=0.25):
    image_np = image_tensor.permute(1, 2, 0).numpy()
    image_np = np.clip(image_np * 255, 0, 255).astype(np.uint8)
    image = Image.fromarray(image_np)
    draw = ImageDraw.Draw(image)

    boxes = output["boxes"]
    labels = output["labels"]
    scores = output["scores"]

    for box, label, score in zip(boxes, labels, scores):
        score_value = float(score.item())

        if score_value < threshold:
            continue

        x1, y1, x2, y2 = box.tolist()
        label_id = int(label.item())

        if label_id == 0:
            label_name = "background"
        else:
            label_name = class_names[label_id - 1]

        text = f"{label_name} {score_value:.2f}"

        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
        draw.text((x1, max(0, y1 - 15)), text, fill="red")

    return image


plt.figure(figsize=(16, 8))

n_show = min(8, len(ssd_prediction_images))

for i in range(n_show):
    img = draw_ssd_prediction(
        ssd_prediction_images[i],
        ssd_prediction_outputs[i],
        class_names,
        threshold=0.25
    )

    plt.subplot(2, 4, i + 1)
    plt.imshow(img)
    plt.title(f"SSD Image {i + 1}")
    plt.axis("off")

plt.suptitle("SSD300 Custom Model Test Predictions")
plt.tight_layout()
plt.savefig("images/tutorial_09_ssd_test_predictions.png", dpi=300, bbox_inches="tight")
plt.show()


<Figure size 1600x800 with 8 Axes>

## 18. Comparison Summary

In [19]:
comparison_df = pd.DataFrame([
    {
        "Model": "Simple SSD from scratch",
        "Purpose": "Architecture/output shape demonstration",
        "Training": "Dummy input only",
        "Output": "Class predictions + box predictions"
    },
    {
        "Model": "Simple YOLO from scratch",
        "Purpose": "Architecture/output shape demonstration",
        "Training": "Dummy input only",
        "Output": "Grid-based detection tensor"
    },
    {
        "Model": "YOLOv8n pretrained + custom trained",
        "Purpose": "Train/test on custom dataset",
        "Training": f"{YOLO_EPOCHS} epochs",
        "Output": "Custom class boxes for Ororon/furina"
    },
    {
        "Model": "SSD300 VGG16 pretrained + custom trained",
        "Purpose": "Train/test on custom dataset",
        "Training": f"{SSD_EPOCHS} epochs",
        "Output": "Custom class boxes for Ororon/furina"
    }
])

comparison_df


,Model,Purpose,Training,Output
0,Simple SSD from scratch,Architecture/output shape demonstration,Dummy input only,Class predictions + box predictions
1,Simple YOLO from scratch,Architecture/output shape demonstration,Dummy input only,Grid-based detection tensor
2,YOLOv8n pretrained + custom trained,Train/test on custom dataset,20 epochs,Custom class boxes for Ororon/furina
3,SSD300 VGG16 pretrained + custom trained,Train/test on custom dataset,5 epochs,Custom class boxes for Ororon/furina


In [20]:
def save_dataframe_as_image(df, filename, title=None, fontsize=8):
    fig_width = max(8, len(df.columns) * 2.0)
    fig_height = max(2.5, len(df) * 0.5 + 1.2)

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=df.values,
        colLabels=df.columns,
        cellLoc="center",
        loc="center"
    )

    table.auto_set_font_size(False)
    table.set_fontsize(fontsize)
    table.scale(1, 1.4)

    if title:
        ax.set_title(title, fontsize=12, pad=12)

    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()


save_dataframe_as_image(
    comparison_df,
    "images/tutorial_09_ssd_yolo_comparison_table.png",
    title="SSD and YOLO Model Comparison",
    fontsize=7
)


<Figure size 800x320 with 1 Axes>

# Final Observations

## SSD

The SSD model uses a backbone feature extractor and multiple prediction heads.  
Each prediction head outputs class scores and bounding-box coordinates.

## YOLO

The YOLO-style model predicts detections from a grid-based output tensor.  
Each grid cell predicts bounding boxes, confidence scores, and class probabilities.

## Custom Dataset

The dataset from Tutorial 08 was reused.  
It contains manually labeled bounding boxes for Ororon and furina.

## Pretrained YOLO

YOLOv8n was fine-tuned on the custom dataset and tested on the test split.

## Pretrained SSD

SSD300 VGG16 was adapted for the custom number of classes and trained on the same dataset.

## Key Learning

SSD and YOLO are both object detection methods, but they organize predictions differently.  
SSD uses multi-scale feature maps and anchor boxes, while YOLO uses grid-based prediction.

Both models require bounding-box labeled data for custom object detection.
